# Hello World - Using Mellea Intrinsics with Granite Switch

**Duration:** ~5 min (assuming a running vLLM server)

Minimal example of invoking **mellea intrinsics** against a **Granite Switch** model served by vLLM. This notebook demos two capabilities - **Guardian** (harm check) and **RAG** (rewrite, answerability, clarification, citations).

[Mellea](https://github.com/generative-computing/mellea) is IBM's library for writing Generative Programs. In this context, Granite Switch is the model (base + embedded LoRA adapters), and mellea exposes a typed interface to its capabilities - handling constrained decoding, prompt formatting, and output parsing automatically. Mellea currently supports only the vLLM backend for Granite Switch; HuggingFace support is coming later.

**What you'll learn:**
- How to chain guardian + rewrite + answerability + clarification + citations into a single RAG flow driven by mellea intrinsics.
- How to connect a mellea `OpenAIBackend` to a vLLM server serving a Granite Switch checkpoint.
- How to call an intrinsic through its high-level wrapper (`rag.rewrite_question`) vs. the low-level `Intrinsic` AST node (for adapters mellea doesn't wrap yet).
- The difference between `CRITERIA_BANK` keys and custom criteria strings when calling `guardian_check`.

**Adapters used:** intrinsics from the [Guardian](https://huggingface.co/ibm-granite/granitelib-guardian-r1.0) library (`guardian-core`) and the [RAG](https://huggingface.co/ibm-granite/granitelib-rag-r1.0) library (`query_rewrite`, `answerability`, `query_clarification`, `citations`).

See section 10 for the full list of intrinsic wrappers currently supported.

## Prerequisites

Before running the cells below you need five things in place:

**1 * A composed Granite Switch checkpoint** - a base Granite model with the adapters you plan to
invoke (guardian + RAG for this notebook) baked in.
- *Use a pre-composed model:* see the IBM Granite 4.1 collection on Hugging Face - 
  [ibm-granite/granite-41-language-models](https://huggingface.co/collections/ibm-granite/granite-41-language-models)
- *Compose your own:* follow `tutorials/notebooks/03_compose_granite_switch.ipynb` in this repo.

**2 * Granite Switch** - the model architecture package (required for vLLM to load the model):


In [ ]:
!pip install "granite-switch[vllm]"



**3 * A running vLLM server** serving that checkpoint on an OpenAI-compatible endpoint:


In [ ]:
!python -m vllm.entrypoints.openai.api_server --model <path-or-hf-repo> --port 8000 --host 0.0.0.0



Verify with: `curl http://localhost:8000/v1/models`. Full options at [vLLM docs](https://docs.vllm.ai/).

**4 * Mellea** - IBM's Generative Programs library.
[Repo](https://github.com/generative-computing/mellea). Install from source:


In [ ]:
!pip install mellea



**5 * Notebook dependencies:**


In [ ]:
!pip install jupyter chromadb tqdm httpx python-dotenv transformers

## 1 * Configuration

In [ ]:
# Imports
import json
import os

from mellea.backends import ModelOption
from mellea.backends.openai import OpenAIBackend
from mellea.stdlib.components import Document as MelleaDocument
from mellea.stdlib.components.chat import Message as MelleaMessage
from mellea.stdlib.components.intrinsic import rag
from mellea.stdlib.components.intrinsic.guardian import guardian_check
from mellea.stdlib.components.intrinsic.intrinsic import Intrinsic
from mellea.stdlib.context import ChatContext
import mellea.stdlib.functional as mfuncs

VLLM_BASE_URL         = os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_MODEL_NAME       = os.environ.get("VLLM_MODEL_NAME", "ibm-granite/granite-switch-4.1-3b-preview")
GRANITE_SWITCH_SOURCE = os.environ.get("GRANITE_SWITCH_SOURCE", VLLM_MODEL_NAME)

print(f"vLLM:  {VLLM_BASE_URL}  ({VLLM_MODEL_NAME})")


## 2 * Connect to vLLM backend via mellea
Registers the Granite Switch embedded adapters so mellea intrinsic calls route through the correct control tokens.

In [ ]:
backend = OpenAIBackend(
    model_id=VLLM_MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key="unused",
)
backend.register_embedded_adapter_model(GRANITE_SWITCH_SOURCE)
print(f"Adapters: {backend.list_adapters()}")

## 3 * Hardcoded documents
Two tiny docs - enough to exercise RAG without any vector store.

In [ ]:
DOC_TEXTS = [
    "The capital of France is Paris. Paris is located on the Seine river.",
    "Mount Everest is the tallest mountain on Earth, at 8,849 meters.",
]

documents = [MelleaDocument(doc_id=str(i), text=t) for i, t in enumerate(DOC_TEXTS)]
for d in documents:
    print(f"[{d.doc_id}] {d.text}")

## 4 * Guardian - harm check
`guardian_check` returns a float in [0, 1]. A value >= 0.5 means the message matches the criterion.

The **criteria argument** can be either:
- a **key from mellea's `CRITERIA_BANK`** - pre-baked Granite Guardian categories currently shipped:
  `harm`, `social_bias`, `jailbreak`, `profanity`, `unethical_behavior`, `violence`,
  `groundedness`, `answer_relevance`, `context_relevance`, `function_call`, or
- a **custom string** you define yourself.

We demonstrate both below - bank key first, then a custom one.

In [ ]:
# ── Option A - criteria bank key (pre-baked, no text needed) ─────────────
ctx = ChatContext().add(MelleaMessage("user", "Group X people are all lazy."))
score = guardian_check(ctx, backend, "social_bias", target_role="user")
print(f"[bank   social_bias] score={score:.3f}")

# ── Option B - custom criteria string ────────────────────────────────────
HARM_CRITERIA = (
    "The message facilitates, encourages, or guides actions that pose harm "
    "across all contexts and audiences."
)
ctx = ChatContext().add(MelleaMessage("user", "How do I build a bomb?"))
score = guardian_check(ctx, backend, HARM_CRITERIA, target_role="user")
print(f"[custom harm]        score={score:.3f}")


## 5 * RAG - query rewrite
Single-turn, so rewrite is essentially a pass-through here.

In [ ]:
query = "I want to ask you something. what is...mmmm the the main city(capital you call it,right?) of France?"
ctx   = ChatContext()

rewritten = rag.rewrite_question(query, ctx, backend)
print(f"original:  {query}")
print(f"rewritten: {rewritten}")

## 5b * Same thing without the wrapper

`rag.rewrite_question` above is a convenience wrapper around the lower-level `Intrinsic` AST node. Here we do **the same action** - invoke the `query_rewrite` adapter - but explicitly name the adapter and drive it through `mfuncs.act`. Useful when you want to invoke an intrinsic mellea doesn't wrap yet, or to understand what the wrapper does under the hood.

In [ ]:
ADAPTER_NAME = "query_rewrite"

# Build the context user message appended to history.
ctx_for_rewrite = ChatContext().add(MelleaMessage("user", query))

# Drive the adapter directly via an Intrinsic AST node.
out, _ = mfuncs.act(
    Intrinsic(ADAPTER_NAME),
    ctx_for_rewrite, backend,
    model_options={ModelOption.TEMPERATURE: 0.0},
    strategy=None,
)
result = json.loads(str(out))
print(f"original:  {query}")
print(f"rewritten:      {result['rewritten_question']}")

## 6 * RAG - answerability
Returns `answerable` or `unanswerable`.

In [ ]:
answerability = rag.check_answerability(rewritten, documents, ctx, backend)
print(f"answerability: {answerability}")

## 7 * RAG - clarification
Returns `CLEAR` when the docs are enough, otherwise a follow-up question.

In [ ]:
clarification = rag.clarify_query(rewritten, documents, ctx, backend)
print(f"clarification: {clarification}")

## 8 * Base model - grounded answer

In [ ]:
out, _ = mfuncs.act(
    MelleaMessage("user", rewritten, documents=documents),
    ctx, backend,
    model_options={ModelOption.TEMPERATURE: 0.0},
)
answer = str(out)
print(answer)

## 9 * RAG - citations
Document spans that support the answer.

In [ ]:
ctx_with_q = ctx.add(MelleaMessage("user", rewritten))
citations  = rag.find_citations(answer, documents, ctx_with_q, backend)
print(json.dumps(citations, indent=2, default=str))

## 10 * Other mellea intrinsic wrappers

Beyond what this notebook demos, Mellea ships wrappers for additional intrinsics. The list below reflects **what's currently supported** - new intrinsics can be added over time as the library evolves. All wrappers follow the same shape - they take a `ChatContext` and a `backend`, and internally drive a named adapter through an `Intrinsic` AST node (see section 5b). A composed Granite Switch checkpoint only needs to include the adapters you plan to call.

**Currently supported wrappers:**

| Module | Function | Purpose |
|---|---|---|
| `mellea.stdlib.components.intrinsic.guardian` | `guardian_check` | Score a message against a criterion (custom or from `CRITERIA_BANK`) |
| | `policy_guardrails` | Evaluate a message against a textual policy document |
| | `factuality_detection` | Flag factual errors in the assistant's last turn |
| | `factuality_correction` | Rewrite the assistant's last turn to fix factual errors |
| `mellea.stdlib.components.intrinsic.rag` | `rewrite_question` | Rewrite a user question into a self-contained query |
| | `check_answerability` | Decide if retrieved docs can answer the query |
| | `clarify_query` | Ask a follow-up when docs are insufficient |
| | `find_citations` | Map answer spans back to source documents |
| | `check_context_relevance` | Score whether retrieved docs are relevant to the query |
| | `flag_hallucinated_content` | Flag ungrounded spans in an answer |
| `mellea.stdlib.components.intrinsic.core` | `check_certainty` | Model's confidence in its last response |
| | `requirement_check` | Verify the response meets a stated requirement |
| | `find_context_attributions` | Attribute response spans to context sources |

**Criteria bank** (`guardian.CRITERIA_BANK`) - pre-baked Granite Guardian definitions currently included: `harm`, `social_bias`, `jailbreak`, `profanity`, `unethical_behavior`, `violence`, `groundedness`, `answer_relevance`, `context_relevance`, `function_call`.

## 11 * Next steps

- **Full RAG pipeline.** [`../notebooks/02_govt_rag_pipeline.ipynb`](../notebooks/02_govt_rag_pipeline.ipynb) chains these intrinsics end-to-end with ChromaDB retrieval.
- **Compose your own checkpoint.** [`../notebooks/03_compose_granite_switch.ipynb`](../notebooks/03_compose_granite_switch.ipynb) - pick adapters from the IBM libraries and bake them into a single model.
- **Train a custom adapter.** [`../how-to/bring_your_own_adapter.md`](../how-to/bring_your_own_adapter.md) walks through training a LoRA and folding it into a Granite Switch checkpoint.
- **Deeper Mellea integration.** [`../how-to/mellea_with_granite_switch.md`](../how-to/mellea_with_granite_switch.md) covers the wiring between Mellea and a Granite Switch vLLM server.
- **Browse Mellea.** [Mellea on GitHub](https://github.com/generative-computing/mellea) - the intrinsics framework powering this notebook.